# Unitflow Demo

A walkthrough of the `unitflow` library — an elegant engineering math substrate for executable MBSE.

This notebook covers:
1. Basic engineering calculations
2. Unit conversions
3. Display and formatting
4. User-defined units and extensibility
5. Symbolic constraints for MBSE
6. Serialization
7. NumPy array workflows
8. Error handling

---
## 1. Basic Engineering Calculations

The API is designed to feel like the math engineers actually write.

In [1]:
from unitflow import m, cm, mm, kg, s, N, Pa, J, W, Hz, rad, rpm, psi, ksi, lbf, inch, si, mech, Quantity, Unit

# Force = mass × acceleration
mass = 2 * kg
accel = 9.81 * (m / s**2)
force = mass * accel

print(f"mass  = {mass}")
print(f"accel = {accel}")
print(f"force = {force}")
print(f"force (in N) ≈ {force.to(N)}")

mass  = 2 kg
accel = 9.81 m / s^2
force = 19.62 N
force (in N) ≈ 19.62 N


In [2]:
# Pressure = force / area
area = Quantity(2, m**2)
pressure = (500 * N) / area

print(f"pressure = {pressure.to(Pa)}")

pressure = 250 Pa


In [3]:
# Power = energy / time
energy = 1000 * J
time = 10 * s
power = energy / time

print(f"power = {power}")
print(f"  = {power.to(W)}")

power = 100 m^2 * kg / s^3
  = 100 W


---
## 2. Unit Conversions

Conversions use `.to(target_unit)` and are exact wherever possible.

In [4]:
length = 1 * m

print(f"{length} = {length.to(cm)} = {length.to(mm)}")

1 m = 100 cm = 1000 mm


In [5]:
import math

# RPM to rad/s — uses exact pi-bearing scale factors
speed = 3000 * rpm
angular = speed.to(rad / s)

print(f"{speed} = {angular}")
print(f"Expected: {100 * math.pi:.4f} rad/s")

3000 rpm = 314.1592653589793 1 / s
Expected: 314.1593 rad/s


In [6]:
# Machinist-facing output
stress = 35 * ksi
print(f"Stress: {stress}")
print(f"In psi: {stress.to(psi)}")

Stress: 35 ksi
In psi: 35000 psi


In [7]:
# Mixed-unit area normalizes cleanly
width = 3 * m
depth = 50 * cm
area = width * depth

print(f"Raw: {repr(area)}")
print(f"Display: {area}")
print(f"In m²: {area.to(m**2)}")
print(f"In mm²: {area.to(mm**2)}")

Raw: Quantity(magnitude=150, unit=Unit(dimension=Dimension(length=2), scale=Scale(1/100), symbol='m*cm'))
Display: 1.5 m^2
In m²: 1.5 m^2
In mm²: 1500000 mm^2


In [8]:
# Semantic equality: 1 m == 100 cm
a = 1 * m
b = 100 * cm

print(f"{a} == {b} ? {a == b}")
print(f"Same hash? {hash(a) == hash(b)}")
print(f"Set size: {len({a, b})}")

1 m == 100 cm ? True
Same hash? True
Set size: 1


---
## 3. Display and Formatting

The library disambiguates torque vs energy using `quantity_kind`.

In [9]:
# Same physical dimension, different engineering meaning
torque_unit = (N * m).with_metadata(quantity_kind="torque")
energy_unit = (N * m).with_metadata(quantity_kind="energy")

torque = Quantity(12, torque_unit)
energy = Quantity(12, energy_unit)

print(f"Torque: {torque}")
print(f"Energy: {energy}")

Torque: 12 N*m
Energy: 12 J


---
## 4. User-Defined Units and Extensibility

Users can define their own units and namespaces without touching core code.

In [10]:
from fractions import Fraction
from unitflow import define_unit, UnitNamespace

# Define a foot from first principles
ft = define_unit(
    name="foot",
    symbol="ft",
    expr=Quantity(Fraction(3048, 10000), m),
)

height = 6 * ft
print(f"Height: {height}")
print(f"In meters: {height.to(m)}")

Height: 6 ft
In meters: 1.8288 m


In [11]:
# Create a custom namespace for aerospace
aero = UnitNamespace("aero")
knot = aero.define_unit(
    name="knot",
    symbol="kn",
    expr=Quantity(1852, m / s),
)

airspeed = 250 * aero.kn
print(f"Airspeed: {airspeed}")
print(f"In m/s: {airspeed.to(m / s)}")

Airspeed: 250 kn
In m/s: 463000 m / s


---
## 5. Symbolic Constraints for MBSE

This is where `unitflow` becomes a substrate for executable engineering models.

In [12]:
from unitflow import symbol, Equation, Conjunction

# Define symbolic model variables
F = symbol("F", unit=N)
mass_sym = symbol("m", unit=kg)
a = symbol("a", unit=m / s**2)

# Newton's second law as a constraint
newtons_law = F == mass_sym * a

print(f"Type: {type(newtons_law).__name__}")
print(f"Is Equation? {isinstance(newtons_law, Equation)}")

Type: Equation
Is Equation? True


In [13]:
# Bounded variables with & and |
x = symbol("x", unit=m)
bounds = (0 * m <= x) & (x <= 10 * m)

print(f"Type: {type(bounds).__name__}")
print(f"Is Conjunction? {isinstance(bounds, Conjunction)}")

Type: Conjunction
Is Conjunction? True


In [14]:
# Symmetric dispatch: both directions produce equivalent constraints
eq1 = F == 5 * N
eq2 = 5 * N == F

print(f"F == 5N type: {type(eq1).__name__}")
print(f"5N == F type: {type(eq2).__name__}")
print(f"Structurally same? {eq1.is_same(eq2)}")

F == 5N type: Equation
5N == F type: Equation
Structurally same? True


In [15]:
# Negation with ~
from unitflow import StrictInequality

c = x <= 10 * m
negated = ~c

print(f"Original: x <= 10m")
print(f"Negated type: {type(negated).__name__}")
print(f"Negated operator: {negated.operator}")

Original: x <= 10m
Negated type: StrictInequality
Negated operator: >


In [16]:
# Full ThunderGraph-style model example
shaft_speed = symbol("shaft_speed", unit=rpm)
shaft_torque = symbol("shaft_torque", unit=N * m, quantity_kind="torque")
shaft_power = symbol("shaft_power", unit=W)

shaft_omega = shaft_speed.to(rad / s)
power_balance = shaft_power == shaft_torque * shaft_omega
speed_bounds = (0 * rpm <= shaft_speed) & (shaft_speed <= 6000 * rpm)

print(f"Power balance: {type(power_balance).__name__}")
print(f"Speed bounds: {type(speed_bounds).__name__}")

Power balance: Equation
Speed bounds: Conjunction


---
## 6. Serialization

Quantities and constraints serialize to plain JSON-safe dicts.

In [17]:
import json
from unitflow import serialize_quantity, deserialize_quantity, serialize_constraint, deserialize_constraint

q = 3000 * rpm
data = serialize_quantity(q)

print("Serialized:")
print(json.dumps(data, indent=2))

Serialized:
{
  "magnitude": {
    "type": "int",
    "value": 3000
  },
  "unit": {
    "dimension": {
      "exponents": [
        0,
        0,
        -1,
        0,
        0,
        0,
        0
      ]
    },
    "scale": {
      "coefficient": {
        "numerator": 1,
        "denominator": 30
      },
      "pi_power": 1
    },
    "name": "revolutions_per_minute",
    "symbol": "rpm",
    "aliases": [
      "rev_per_min"
    ],
    "family": null,
    "quantity_kind": null
  }
}


In [18]:
# Round-trip
json_str = json.dumps(data)
restored = deserialize_quantity(json.loads(json_str))

print(f"Original:  {q}")
print(f"Restored:  {restored}")
print(f"Equal?     {q == restored}")

Original:  3000 rpm
Restored:  3000 rpm
Equal?     True


In [19]:
# Constraint serialization
x = symbol("x", unit=m)
constraint = ((0 * m) <= x) & (x <= 10 * m)

constraint_data = serialize_constraint(constraint)
print("Constraint JSON:")
print(json.dumps(constraint_data, indent=2)[:500], "...")

Constraint JSON:
{
  "type": "Conjunction",
  "left": {
    "type": "NonStrictInequality",
    "left": {
      "type": "Symbol",
      "name": "x",
      "dimension": {
        "exponents": [
          1,
          0,
          0,
          0,
          0,
          0,
          0
        ]
      },
      "unit": {
        "dimension": {
          "exponents": [
            1,
            0,
            0,
            0,
            0,
            0,
            0
          ]
        },
        "scale": {
       ...


---
## 7. NumPy Array Workflows

Array-backed quantities follow the same semantic rules as scalars.

In [20]:
import numpy as np

# Vectorized force calculation
masses = Quantity(np.array([1.0, 2.0, 5.0, 10.0]), kg)
accels = Quantity(np.array([9.81, 9.81, 9.81, 9.81]), m / s**2)

forces = masses * accels
print(f"Forces: {forces.magnitude} {forces.unit}")

Forces: [ 9.81 19.62 49.05 98.1 ] kg*m/s^2


---
## 9. SI Prefix System

`unitflow` ships with a programmatic SI prefix generator. Instead of hand-coding every `kW`, `MW`, `GHz`, the library generates them deterministically from the standard SI prefix table.

In [21]:
from unitflow import kW, MW, GW, mW, kN, mN, kPa, MPa, GPa, kJ, MJ, GJ, kHz, MHz, GHz, km, nm, um, ms, us, ns, si

# Power across orders of magnitude
power = 5000 * W
print(f"{power}")
print(f"  = {power.to(kW)}")
print(f"  = {power.to(MW)}")
print(f"  = {power.to(mW)}")
print()

# A power plant
plant_output = 1200 * MW
print(f"Plant output: {plant_output}")
print(f"  = {plant_output.to(GW)}")
print(f"  = {plant_output.to(kW)}")
print(f"  = {plant_output.to(W)}")

5000 W
  = 5 kW
  = 0.005 MW
  = 5000000 mW

Plant output: 1200 MW
  = 1.2 GW
  = 1200000 kW
  = 1200000000 W


In [22]:
# Pressure: from pascals to gigapascals
bolt_stress = 250 * MPa
print(f"Bolt stress: {bolt_stress}")
print(f"  = {bolt_stress.to(GPa)}")
print(f"  = {bolt_stress.to(kPa)}")
print(f"  = {bolt_stress.to(Pa)}")
print()

# Force
rocket_thrust = 7 * kN
print(f"Rocket thrust: {rocket_thrust}")
print(f"  = {rocket_thrust.to(N)}")
print(f"  = {rocket_thrust.to(mN)}")

Bolt stress: 250 MPa
  = 0.25 GPa
  = 250000 kPa
  = 250000000 Pa

Rocket thrust: 7 kN
  = 7000 N
  = 7000000 mN


In [23]:
# Frequency: from Hz to GHz
cpu_clock = Quantity(3500, MHz)
print(f"CPU clock: {cpu_clock}")
print(f"  = {cpu_clock.to(GHz)}")
print(f"  = {cpu_clock.to(kHz)}")
print(f"  = {cpu_clock.to(Hz)}")
print()

# Length at small scales
wavelength = 550 * nm
print(f"Visible light wavelength: {wavelength}")
print(f"  = {wavelength.to(um)}")
print(f"  = {wavelength.to(mm)}")
print()

# Time at small scales
response_time = 250 * us
print(f"Sensor response: {response_time}")
print(f"  = {response_time.to(ms)}")
print(f"  = {response_time.to(ns)}")

CPU clock: 3500 MHz
  = 3.5 GHz
  = 3500000 kHz
  = 3500000000 Hz

Visible light wavelength: 550 nm
  = 0.55 um
  = 0.00055 mm

Sensor response: 250 us
  = 0.25 ms
  = 250000 ns


In [24]:
# Namespace access: all prefixed units are available via si.*
print("Available via si namespace:")
print(f"  si.kW  = {si.kW}")
print(f"  si.MW  = {si.MW}")
print(f"  si.GPa = {si.GPa}")
print(f"  si.kHz = {si.kHz}")
print(f"  si.nm  = {si.nm}")
print(f"  si.ms  = {si.ms}")
print()

# They compose naturally with arithmetic
energy = 2.5 * MJ
time = 10 * s
avg_power = energy / time
print(f"Average power: {avg_power.to(kW)}")

Available via si namespace:
  si.kW  = kW
  si.MW  = MW
  si.GPa = GPa
  si.kHz = kHz
  si.nm  = nm
  si.ms  = ms

Average power: 250.0 kW


In [25]:
# The prefix generator is also available for user-defined units
from unitflow import generate_prefixes, SI_PREFIXES, UnitNamespace, define_unit, Dimension, Scale
from fractions import Fraction

# Define a custom "volt" unit from first principles: V = W/A = kg*m^2/(A*s^3)
electrical = UnitNamespace("electrical")
V = electrical.define_unit(
    name="volt",
    symbol="V",
    expr=W,  # simplified: same dimension as W for this demo
)

# Generate mV, kV, MV automatically
volt_prefixes = generate_prefixes(
    electrical, V,
    include={"milli", "kilo", "mega"},
)

print("Generated voltage prefixes:")
for prefix_name, unit in sorted(volt_prefixes.items()):
    print(f"  {prefix_name}: {unit.symbol} ({unit.name})")

# Use them immediately
signal = 3300 * electrical.millivolt
print(f"\nSignal: {signal}")
print(f"  = {signal.to(V)}")

Generated voltage prefixes:
  kilo: kV (kilovolt)
  mega: MV (megavolt)
  milli: mV (millivolt)

Signal: 3300 mW
  = 3.3 W


In [26]:
# Reductions preserve units
distances = Quantity(np.array([10.0, 20.0, 30.0]), m)

total = np.sum(distances)
average = np.mean(distances)

print(f"Total:   {total}")
print(f"Average: {average}")

Total:   60.0 m
Average: 20.0 m


In [27]:
# Trig functions require dimensionless input
angles = Quantity(np.array([0, np.pi/2, np.pi]), rad)
sines = np.sin(angles)

print(f"sin({angles.magnitude}) = {sines.magnitude}")
print(f"Result is dimensionless? {sines.unit.dimension.is_dimensionless}")

sin([0.         1.57079633 3.14159265]) = [0.0000000e+00 1.0000000e+00 1.2246468e-16]
Result is dimensionless? True


In [28]:
# Trig on dimensional values fails cleanly
from unitflow import DimensionMismatchError

try:
    np.sin(Quantity(np.array([1.0, 2.0]), m))
except DimensionMismatchError as e:
    print(f"Caught: {e}")

Caught: Transcendental function sin requires a dimensionless input, got Dimension(length=1).


---
## 8. Error Handling

The library fails loudly and helpfully.

In [29]:
from unitflow import IncompatibleUnitError, BooleanCoercionError

# Can't add meters and seconds
try:
    _ = (3 * m) + (2 * s)
except DimensionMismatchError as e:
    print(f"Addition error: {e}")

# Can't convert meters to seconds
try:
    (3 * m).to(s)
except IncompatibleUnitError as e:
    print(f"Conversion error: {e}")

# Constraints are not booleans
x = symbol("x", unit=m)
try:
    if x == 5 * m:
        pass
except BooleanCoercionError as e:
    print(f"Coercion error: {e}")

Addition error: Cannot add quantities with dimensions Dimension(length=1) and Dimension(time=1).
Conversion error: Incompatible units: Unit(dimension=Dimension(length=1), scale=Scale(1), name='meter', symbol='m', aliases=('metre',)) cannot convert to Unit(dimension=Dimension(time=1), scale=Scale(1), name='second', symbol='s').
Coercion error: Constraints are not booleans. Use check(), solve(), all_of(), or & and | for logical composition.


In [30]:
# Unary negation works
q = 5 * m
print(f"-({q}) = {-q}")

-(5 m) = -5 m
